In [6]:
from dataclasses import replace

import tabulate
import pandas as pd

from config.experiment import ExperimentConfig
from config.task import generate_task_configs
from experiments import load_config
from experiments.configstore import embedders
from utils.plot import Metric, PlotFilter, get_metrics, plot_metrics_vs_perturbation, shorten_model_name, strip_hf_org, cleanup_colnames_after_groupby, combine_columns

In [7]:
def get_config(cfg_name: str) -> ExperimentConfig:
    exp_cfg = load_config(cfg_name)

    return ExperimentConfig(
        train=replace(exp_cfg.train, save_folder=exp_cfg.train.save_folder.parent / "paper-multirun"),
        eval=replace(exp_cfg.eval, results_folder=exp_cfg.eval.results_folder.parent / "paper-multirun"),
    )

In [8]:
baselines = {
    "paper_targeted_attacks_oneQ_oneA": "paper_targeted_1-1_neg_baseline",
    "paper_targeted_attacks_multiQ_oneA": "paper_targeted_m-1_neg_baseline",
    "paper_targeted_attacks_multiQ_multiA": "paper_targeted_m-m_neg_baseline",
    "paper_multi_transferability_targeted": "paper_multi_transferability_targeted_neg_baseline",
    "paper_leave_one_out_multi_transferability_targeted": "paper_leave_one_out_multi_transferability_targeted_neg_baseline",
    "paper_non_targeted" :"paper_universal_GT_baseline",
    "paper_multi_transferability": "paper_multi_transferability_GT_baseline",
    "paper_leave_one_out_multi_transferability": "paper_leave_one_out_multi_transferability_GT_baseline",
}

In [9]:
# exp_config = get_config("paper_perturbation_plot")

# fig = plot_metrics_vs_perturbation(
#     exp_config,
#     metrics_to_show=[
#         Metric.RETRIEVAL_ASR_TRAIN,
#         Metric.RETRIEVAL_ASR_TEST,
#         Metric.GENERATION_ASR_EXACT_TRAIN,
#         Metric.GENERATION_ASR_EXACT_TEST,
#         Metric.GENERATION_ACC_EMBED_GT_TRAIN,
#         Metric.GENERATION_ACC_EMBED_GT_TEST,
#     ],
#     ret_topk_idx=0,
#     gen_topk_idx=0,
# )

# from config import OUTPUTS_FOLDER
# fig.savefig(OUTPUTS_FOLDER / "perturbation.pdf", bbox_inches='tight')

In [10]:
# exp_config = get_config("perturbation_plot_targeted")
#
# plot_metrics_vs_perturbation(
#     exp_config,
#     metrics_to_show=[
#         Metric.RETRIEVAL_ASR_TRAIN,
#         Metric.RETRIEVAL_ASR_TEST,
#         Metric.GENERATION_ASR_EXACT_TRAIN,
#         Metric.GENERATION_ASR_EXACT_TEST,
#         Metric.GENERATION_ACC_EMBED_GT_TRAIN,
#         Metric.GENERATION_ACC_EMBED_GT_TEST,
#     ],
#     ret_topk_idx=0,
#     gen_topk_idx=0,
# )

In [11]:
def update_multirow(df, column, row_index, count, value):
    df.loc[row_index - count, column] = f"\multirow{{{count}}}{{*}}{{{value}}}"
    for j in range(count - 1):
         df.loc[row_index - 1 - j, column] = ""

def multirow_columns(df, columns: list[str]):
    for column in columns:
        if column in df.columns:
            prev_val = None
            count = 1
            for i, value in chain(enumerate(df[column])):
                if prev_val == value:
                    count += 1
                else:
                    if count > 1:
                        update_multirow(df, column, i, count, prev_val)
                    count = 1
                    prev_val = value

            if count > 1:
                update_multirow(df, column, len(df[column]), count, prev_val)

In [12]:
from itertools import chain
from typing import Literal

# do_combine_aggs = True
do_aggr = True

def make_all_metric_table(config_name: str, tbl_format: Literal["html", "latex_raw", "latex_booktabs"] = "html", metrics_to_show: list[Metric] | None = None, row_filter=None, at: Literal[1,5] | None = None, name=None, attack_type="", collapse=False):
    if config_name in baselines:
        baseline_config = get_config(baselines[config_name])
        baseline_task_configs = generate_task_configs(baseline_config, include_eval=True)
        baseline_metrics = dict()
        for c in baseline_task_configs:
            metrics, _ = get_metrics(
                exp_config= baseline_config,
                task_config=c,
                metrics_to_show=list(set(metrics_to_show) & {Metric.GENERATION_FPR_EMBED_TARGETED_TEST, Metric.GENERATION_ACC_EMBED_GT_TEST}),
            )
            baseline_metrics[c.create_hash_string()+c.get_transferability_file_suffix()] = metrics

    exp_config = get_config(config_name)

    if metrics_to_show is None:
        metrics_to_show = [m for m in Metric]

    task_configs = generate_task_configs(exp_config, include_eval=True)

    table = []
    for task_config in task_configs:
        row = {
            "dataset": strip_hf_org(task_config.ds_name),
            "image index": task_config.chosen_index,
        }
        if exp_config.eval.test_generative_attacks:
            row["embedder"] = task_config.test_generative_attack
        else:
            row["embedder"] = shorten_model_name(task_config.model_name_embs[0]) if len(task_config.model_name_embs) == 1 else "+".join([shorten_model_name(m) for m in task_config.model_name_embs])
            if task_config.vlm:
                row["vlm"] = shorten_model_name(task_config.vlm.models[0]) if len(task_config.vlm.models) == 1 else "+".join([shorten_model_name(m) for m in task_config.vlm.models])
                if len(exp_config.train.vlm.gen_topk_list) > 1:
                    row["vlm topk"] = task_config.vlm.gen_topk
        if len(exp_config.train.emb_train_loss_type_list) > 1:
            row["emb train loss"] = task_config.emb_train_loss_type
        if len(exp_config.train.attack_mask_list) > 1:
            row["attack mask"] = task_config.attack_mask.name
        if task_config.eval_emb_name:
            row["eval emb"] = shorten_model_name(task_config.eval_emb_name)
        if task_config.eval_vlm_name:
            row["eval vlm"] = shorten_model_name(task_config.eval_vlm_name)

        if task_config.judge:
            row["judge"] = shorten_model_name(task_config.judge.model_name)
        if task_config.eval_jdg_name:
            row["eval judge"] = shorten_model_name(task_config.eval_jdg_name)
        metrics, _ = get_metrics(
            exp_config=exp_config,
            task_config=task_config,
            metrics_to_show=metrics_to_show,
        )
        if config_name in baselines:
            try:
                bm = baseline_metrics[task_config.create_hash_string()]
            except KeyError:
                bm = baseline_metrics[task_config.create_hash_string()+task_config.get_transferability_file_suffix()]
            for key, baseline_value in bm.items():
                metrics[f"Δ{key}"] = metrics[key] - baseline_value

        row.update(metrics)
        table.append(row)
    if row_filter:
        table = [row for row in table if row_filter(row)]
    df = pd.DataFrame(table)

    if "Recall-A@1" in df.columns:
        df["Recall-A@1"] = df["Recall-A@1"] - df["Recall-B@1"]
        df["Recall-A@5"] = df["Recall-A@5"] - df["Recall-B@5"]
        df.rename(columns={"Recall-A@1":"ΔRecall-A@1", "Recall-A@5":"ΔRecall-A@5",}, inplace=True)

    if at is not None:
        df = df.filter(regex=f"(@(-1|{at}))|[a-z]$")


    # aggregate similar settings and show mean
    if do_aggr:
        grouping_columns = ['dataset', 'embedder']
        if not exp_config.eval.test_generative_attacks:
            grouping_columns += ['vlm']
        if "eval emb" in df.columns:
            grouping_columns += ['eval emb', 'eval vlm']

        if "eval judge" in df.columns:
            grouping_columns += ['eval judge']

        if "judge" in df.columns:
            grouping_columns += ['judge']


        excluded_columns = grouping_columns + ['image index']
        aggregate_columns = [col for col in df.columns.tolist() if col not in excluded_columns]

        agg_dict = {col: ['mean', 'max'] for col in aggregate_columns}
        df = df.groupby(['dataset'] if collapse else grouping_columns)
        df = df.agg(agg_dict).reset_index()
        df.columns = cleanup_colnames_after_groupby(df.columns)

        # if do_combine_aggs: combine_columns(df, aggregate_columns)

    try:
        if df["eval emb"].equals(df["embedder"]):
            df = df.drop(columns=["eval emb"])
    except KeyError:
        pass
    try:
        if df["eval vlm"].equals(df["vlm"]):
            df = df.drop(columns=["eval vlm"])
    except KeyError:
        pass

    if tbl_format.startswith("latex"):
        try:
            df = df.drop(columns=["dataset"])
        except KeyError:
            pass
        # replace with commands in overleaf
        df = df.replace({
            "CLIP-L": "\cliplargeShort",
            "ColPali": "\colpaliShort",
            "GME-Qwen2-VL-2B": "\gmeShort",
            "InternVL3-2B": "\gmeShort",
            "Qwen2.5-VL-3B": "\qwenVLShort",
            "SmolVLM": "\smolVLMShort",
        })
        df.insert(0, "Attack type", "")
        df.loc[0,"Attack type"] = f"\multirow{{{df.shape[0]}}}{{*}}{{{attack_type}}}"
        # replace with multirow commands
        multirow_columns(df, ["embedder", "vlm", "eval emb", "eval vlm"])
        if collapse:
            df.insert(1, "embedder", "ANY")
            df.insert(2, "vlm", "ANY")

    drop_columns= [
        f"{extra}{metric}@{num} max"
        for metric in [
            Metric.RETRIEVAL_ASR_TARGETED,
            Metric.RETRIEVAL_FPR_TARGETED_TEST,
            Metric.GENERATION_FPR_EMBED_TARGETED_TEST,
            Metric.RETRIEVAL_RECALL_BEFORE,
            Metric.RETRIEVAL_RECALL_AFTER,
            Metric.GENERATION_ACC_EMBED_GT_TEST,
        ]
        for num in [-1, 1, 5]
        for extra in ["", "Δ"]
    ]
    for column in drop_columns:
        if column in df.columns:
            df.drop(column, axis=1, inplace=True)

    tabulate_table = tabulate.tabulate(df, headers="keys", tablefmt=tbl_format, showindex="never")

    if tbl_format.startswith("latex"):
        with open(f'tables/{name or config_name}.tex', 'w') as f:
            f.write(tabulate_table)
        return None
    return tabulate_table

In [13]:
# ================================================================= LATEX TABLES =======================================================

In [ ]:
# TABLE 1 - Targeted Setting I - one to one
make_all_metric_table("paper_targeted_attacks_oneQ_oneA", tbl_format = "latex_raw", row_filter=PlotFilter.CONDITION_SAME_MODELS, metrics_to_show=PlotFilter.ALL_METRICS_TARGETED, name="table1_1", attack_type="White-box")
make_all_metric_table("paper_multi_transferability_targeted", tbl_format = "latex_raw", metrics_to_show=PlotFilter.ALL_METRICS_TARGETED, name ="table1_2", attack_type="Lucky Candidate-Set")
make_all_metric_table("paper_leave_one_out_multi_transferability_targeted", metrics_to_show=PlotFilter.ALL_METRICS_TARGETED, row_filter=PlotFilter.CONDITION_NOT_INCLUDES_MODELS, name ="table1_3", tbl_format = "latex_raw", attack_type="Unlucky Candidate-Set", collapse=True)
make_all_metric_table("paper_targeted_attacks_oneQ_oneA", tbl_format = "latex_raw", row_filter=PlotFilter.CONDITION_NOT_INCLUDES_MODELS, metrics_to_show=PlotFilter.ALL_METRICS_TARGETED, name="table1_4", attack_type="Direct Transferability", collapse=True)
make_all_metric_table("paper_targeted_attacks_oneQ_oneA", tbl_format = "latex_raw", metrics_to_show=PlotFilter.ALL_METRICS_TARGETED, row_filter=PlotFilter.CONDITION_INCLUDES_VLM, name="table1_5", attack_type="Component-wise Transferability (VLM)", collapse=True)
make_all_metric_table("paper_targeted_attacks_oneQ_oneA", tbl_format = "latex_raw", metrics_to_show=PlotFilter.ALL_METRICS_TARGETED, row_filter=PlotFilter.CONDITION_INCLUDES_EMB, name="table1_6", attack_type="Component-wise Transferability (Embedder)", collapse=True)
make_all_metric_table("paper_GPT_targeted_attacks_oneQ_oneA", tbl_format = "latex_raw", metrics_to_show=PlotFilter.ALL_METRICS_TARGETED, name="table1_7", attack_type="Generative")

In [ ]:
# TABLE 2 - Targeted Setting II - many to one
make_all_metric_table("paper_targeted_attacks_multiQ_oneA",tbl_format = "latex_raw", metrics_to_show=PlotFilter.ALL_METRICS_TARGETED, name="table2_1", attack_type="White-box")
make_all_metric_table("paper_GPT_targeted_attacks_multiQ_oneA", tbl_format = "latex_raw", metrics_to_show=PlotFilter.ALL_METRICS_TARGETED, name="table2_2", attack_type="Generative")

In [ ]:
# TABLE 3 - Targeted Setting III - many to many
make_all_metric_table("paper_targeted_attacks_multiQ_multiA",tbl_format = "latex_raw", metrics_to_show=PlotFilter.ALL_METRICS_TARGETED, name="table3_1", attack_type="White-box")
make_all_metric_table("paper_GPT_targeted_attacks_multiQ_multiA", tbl_format = "latex_raw", metrics_to_show=PlotFilter.ALL_METRICS_TARGETED, name="table3_2", attack_type="Generative")

In [14]:
# TABLE 4 - Universal
make_all_metric_table("paper_non_targeted", tbl_format = "latex_raw", metrics_to_show=PlotFilter.ALL_METRICS_UNTARGETED, row_filter=PlotFilter.CONDITION_SAME_MODELS, name="table4_1", attack_type="White-box")
make_all_metric_table("paper_multi_transferability", tbl_format = "latex_raw", metrics_to_show=PlotFilter.ALL_METRICS_UNTARGETED, name ="table4_2", attack_type="Lucky Candidate-Set")
make_all_metric_table("paper_leave_one_out_multi_transferability", metrics_to_show=PlotFilter.ALL_METRICS_UNTARGETED, row_filter=PlotFilter.CONDITION_NOT_INCLUDES_MODELS, name ="table4_3", tbl_format = "latex_raw", attack_type="Unlucky Candidate-Set", collapse=True)
make_all_metric_table("paper_non_targeted", tbl_format = "latex_raw", metrics_to_show=PlotFilter.ALL_METRICS_UNTARGETED, row_filter=PlotFilter.CONDITION_NOT_INCLUDES_MODELS, name="table4_4", attack_type="Direct Transferability")

make_all_metric_table("paper_non_targeted", tbl_format = "latex_raw", metrics_to_show=PlotFilter.ALL_METRICS_UNTARGETED, row_filter=PlotFilter.CONDITION_INCLUDES_VLM,name="table4_5", attack_type="Component-wise Transferability (VLM)", collapse=True)
make_all_metric_table("paper_non_targeted", tbl_format = "latex_raw", metrics_to_show=PlotFilter.ALL_METRICS_UNTARGETED, row_filter=PlotFilter.CONDITION_INCLUDES_EMB, name="table4_6", attack_type="Component-wise Transferability (Embedder)", collapse=True)
make_all_metric_table("paper_GPT_non_targeted", tbl_format = "latex_raw", metrics_to_show=PlotFilter.ALL_METRICS_UNTARGETED, name="table4_7", attack_type="Generative")

In [ ]:
# ================================================================= HTML TABLES =======================================================

In [ ]:
make_all_metric_table("paper_non_targeted", metrics_to_show=PlotFilter.ALL_METRICS_UNTARGETED, row_filter=PlotFilter.CONDITION_SAME_MODELS, at=1)

In [ ]:
make_all_metric_table("paper_targeted_attacks_oneQ_oneA",tbl_format = "html", row_filter=PlotFilter.CONDITION_SAME_MODELS, metrics_to_show=PlotFilter.ALL_METRICS_TARGETED)

In [ ]:
make_all_metric_table("paper_targeted_attacks_multiQ_oneA", metrics_to_show=PlotFilter.ALL_METRICS_TARGETED)

In [ ]:
make_all_metric_table("paper_targeted_attacks_multiQ_multiA", metrics_to_show=PlotFilter.ALL_METRICS_TARGETED)

In [ ]:
make_all_metric_table("paper_judge_defence", metrics_to_show=PlotFilter.METRICS_TEST_JUDGE)

In [ ]:
make_all_metric_table("paper_judge_defence_adapt", metrics_to_show=PlotFilter.METRICS_TEST_JUDGE)

In [ ]:
make_all_metric_table("paper_judge_defence_targeted", metrics_to_show=PlotFilter.METRICS_TEST_JUDGE)

In [ ]:
make_all_metric_table("paper_judge_defence_targeted_adapt", metrics_to_show=PlotFilter.METRICS_TEST_JUDGE)

In [ ]:
make_all_metric_table("paper_multi_transferability", metrics_to_show=PlotFilter.ALL_METRICS_UNTARGETED)

In [ ]:
make_all_metric_table("paper_multi_transferability_targeted", metrics_to_show=PlotFilter.ALL_METRICS_TARGETED)

In [ ]:
make_all_metric_table("paper_leave_one_out_multi_transferability", metrics_to_show=PlotFilter.ALL_METRICS_UNTARGETED)

In [ ]:
make_all_metric_table("paper_leave_one_out_multi_transferability_targeted", metrics_to_show=PlotFilter.ALL_METRICS_TARGETED, row_filter=PlotFilter.CONDITION_NOT_INCLUDES_MODELS)

In [ ]:
make_all_metric_table("paper_targeted_attacks_oneQ_oneA",tbl_format = "html", metrics_to_show=PlotFilter.ALL_METRICS_TARGETED, row_filter=PlotFilter.CONDITION_INCLUDES_EMB)

In [ ]:
make_all_metric_table("paper_targeted_attacks_oneQ_oneA",tbl_format = "html", metrics_to_show=PlotFilter.ALL_METRICS_TARGETED, row_filter=PlotFilter.CONDITION_INCLUDES_VLM)

In [ ]:
make_all_metric_table("paper_topk_context", metrics_to_show=PlotFilter.METRICS_TOPK)

In [ ]:
make_all_metric_table("paper_topk_context_targeted", metrics_to_show=PlotFilter.METRICS_TOPK_TARGETED)

In [ ]:
make_all_metric_table("paper_GPT_non_targeted", metrics_to_show=PlotFilter.ALL_METRICS_UNTARGETED)

In [ ]:
make_all_metric_table("paper_GPT_targeted_attacks_oneQ_oneA", metrics_to_show=PlotFilter.ALL_METRICS_TARGETED)

In [ ]:
make_all_metric_table("paper_GPT_targeted_attacks_multiQ_oneA", metrics_to_show=PlotFilter.ALL_METRICS_TARGETED)

In [ ]:
make_all_metric_table("paper_GPT_targeted_attacks_multiQ_multiA", metrics_to_show=PlotFilter.ALL_METRICS_TARGETED)

In [ ]:
make_all_metric_table("paper_copali_ab", metrics_to_show=PlotFilter.METRICS_COLPALI)

In [ ]:
make_all_metric_table("paper_copali_ab_cpoiT", metrics_to_show=PlotFilter.METRICS_COLPALI)

In [ ]:
make_all_metric_table("paper_defences", metrics_to_show=PlotFilter.METRICS_TOPK)

In [ ]:
make_all_metric_table("paper_targeted_defences", metrics_to_show=PlotFilter.METRICS_TOPK_TARGETED)

In [ ]:
# make_all_metric_table("mask_attack")